In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv('RAYA HSI.csv')

data.tail()

,Date,State,City,Population,EnergySource,WaterConsumption,SGD %,AQI,AQI_Bucket,Waste Type,Disposal Method,Recycling Rate (%),Cost of Waste Management(?/Ton)
20085,31-12-2024,Uttar Pradesh,Noida,712593,Hydro,146.54,20.86,111,Moderate,Organic,Landfill,52.60,2606.29
20086,31-12-2024,Uttar Pradesh,Ghaziabad,1740165,Solar,187.39,21.25,198,Moderate,Organic,Composting,46.46,4413.87
20087,31-12-2024,Uttar Pradesh,Ghazipur,141519,Wind,143.42,13.14,340,Very Poor,E-Waste,Incineration,33.94,2763.71
20088,31-12-2024,Uttar Pradesh,Lucknow,3214849,Wind,177.43,11.57,108,Moderate,Plastic,Landfill,29.81,3785.70
20089,31-12-2024,Uttar Pradesh,Meerut,1417862,Wind,100.29,18.78,86,Satisfactory,Plastic,Composting,15.67,2301.46


In [2]:
data.isnull().sum()

,0
Date,0
State,0
City,0
Population,0
EnergySource,0
WaterConsumption,0
SGD %,0
AQI,0
AQI_Bucket,0
Waste Type,0


In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Select features for clustering
features = data[['Population',
 'WaterConsumption', 'SGD %', 'AQI',
 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']]


# -----------------------------
# Scale features — very important for K-Means
# -----------------------------
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

In [4]:
# -----------------------------
# Find optimal k with Elbow Method
# -----------------------------
inertia = []
k_range = range(2, 10)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(features_scaled)
    inertia.append(kmeans.inertia_)  # Distortion / SSE

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=list(k_range),
        y=inertia,
        mode='lines+markers',
        marker=dict(color='royalblue', size=8),
        line=dict(width=2),
        name='Inertia'
    )
)

fig.update_layout(
    title='Elbow Method For Optimal k',
    xaxis_title='Number of clusters (k)',
    yaxis_title='Inertia (SSE)',
    xaxis=dict(tickmode='linear'),
    template='plotly_white',
    width=800,
    height=500
)

fig.show()

In [5]:
# Suppose you choose k= from the elbow
optimal_k = 5

kmeans_final = KMeans(n_clusters=optimal_k, random_state=42)
data['HSI_Type'] = kmeans_final.fit_predict(features_scaled)

print(data.groupby('HSI_Type')[['Population', 'WaterConsumption', 'SGD %', 'AQI', 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']].mean())

            Population  WaterConsumption      SGD %         AQI  \
HSI_Type                                                          
0         1.001271e+06        121.884530  14.659392  239.818715   
1         9.908597e+05        140.135015  15.092110  240.954000   
2         1.007336e+06        205.885331  14.788722  241.615686   
3         9.580294e+05        191.720617  15.206336  252.293574   
4         3.200275e+06        165.334372  15.102571  244.631606   

          Recycling Rate (%)  Cost of Waste Management(?/Ton)  
HSI_Type                                                       
0                  27.655551                      3620.926269  
1                  48.085197                      2184.774375  
2                  27.461226                      2482.162625  
3                  46.454814                      4052.992635  
4                  37.416759                      3096.931110  


In [6]:
cluster_avg = data.groupby("HSI_Type")[['Population', 'WaterConsumption', 'SGD %', 'AQI', 'Recycling Rate (%)', 'Cost of Waste Management(?/Ton)']].mean().sort_values(by='Population')
ordered_clusters = cluster_avg.index.tolist()

In [7]:
ordered_clusters

[3, 1, 0, 2, 4]

In [8]:
# Make a mapping
segment_names = ["Sustainable", "Low Sustainable", "Critical (Unsustainable)", "Moderately Sustainable", "Highly Sustainable"]
cluster_to_label = {cluster: segment_names[i] for i, cluster in enumerate(ordered_clusters)}

# Apply mapping
data["HSI_Label"] = data["HSI_Type"].map(cluster_to_label)

print(data[["City","HSI_Type", "HSI_Label"]].head())
print('--' * 25)
print(data[["HSI_Type", "HSI_Label"]].value_counts())

        City  HSI_Type           HSI_Label
0      Noida         1     Low Sustainable
1  Ghaziabad         1     Low Sustainable
2   Ghazipur         3         Sustainable
3    Lucknow         4  Highly Sustainable
4     Meerut         3         Sustainable
--------------------------------------------------
HSI_Type  HSI_Label               
3         Sustainable                 4217
0         Critical (Unsustainable)    4093
1         Low Sustainable             4000
4         Highly Sustainable          3955
2         Moderately Sustainable      3825
Name: count, dtype: int64


In [ ]:
df_numeric = data.select_dtypes(include=['number'])
corr = df_numeric.corr()

import plotly.express as px

fig = px.imshow(
    corr,
    text_auto=".2f",
    color_continuous_scale='Teal',
    title="Correlation Heatmap (Numeric Columns Only)"
)

fig.update_layout(
    width=850,
    height=600
)

fig.show()


In [9]:
data['Date'] = pd.to_datetime(data['Date'], format='%d-%m-%Y')

monthly_df = (
    data.groupby(["City", pd.Grouper(key="Date", freq="ME")])["HSI_Type"]
      .mean()
      .reset_index()
)

In [10]:
from prophet import Prophet

forecast_results = {}

for city in monthly_df["City"].unique():

    city_df = monthly_df[monthly_df["City"] == city][["Date", "HSI_Type"]]
    city_df = city_df.rename(columns={"Date": "ds", "HSI_Type": "y"})

    model = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False
    )

    model.fit(city_df)

    # 5 years = 60 months
    future = model.make_future_dataframe(periods=60, freq="ME")
    forecast = model.predict(future)

    forecast_results[city] = (model, forecast)

In [11]:
import plotly.graph_objects as go

# Sustainability labels
hsi_labels = {
    0: "Critical (Unsustainable)",
    1: "Low Sustainable",
    2: "Moderately Sustainable",
    3: "Sustainable",
    4: "Highly Sustainable"
}

for city, (model, forecast) in forecast_results.items():

    fig = go.Figure()

    # ----------------------------
    # Historical actual HSI level
    # ----------------------------
    fig.add_trace(go.Scatter(
        x=model.history["ds"],
        y=model.history["y"],
        mode="lines+markers",
        name="Actual HSI Level",
        line=dict(width=2)
    ))

    # ----------------------------
    # Continuous forecast (trend)
    # ----------------------------
    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["yhat"],
        mode="lines",
        name="Forecast Trend",
        line=dict(dash="dash")
    ))

    # ----------------------------
    # Discrete forecasted HSI level
    # ----------------------------
    forecast["HSI_Level"] = forecast["yhat"].round().clip(0, 4)

    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["HSI_Level"],
        mode="lines",
        name="Forecasted HSI Category",
        line=dict(width=3)
    ))

    # ----------------------------
    # Confidence band
    # ----------------------------
    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["yhat_upper"],
        line=dict(width=0),
        showlegend=False
    ))

    fig.add_trace(go.Scatter(
        x=forecast["ds"],
        y=forecast["yhat_lower"],
        fill="tonexty",
        line=dict(width=0),
        fillcolor="rgba(150,150,150,0.2)",
        name="Forecast Uncertainty"
    ))

    # ----------------------------
    # Sustainability bands
    # ----------------------------
    for level, label in hsi_labels.items():
        fig.add_hline(
            y=level,
            line_dash="dot",
            annotation_text=label,
            annotation_position="left"
        )

    # ----------------------------
    # Layout
    # ----------------------------
    fig.update_layout(
        title=f"5-Year City-wise Human Sustainability Forecast — {city}",
        xaxis_title="Year",
        yaxis_title="Sustainability Level",
        yaxis=dict(
            tickmode="array",
            tickvals=list(hsi_labels.keys()),
            ticktext=list(hsi_labels.values()),
            range=[-0.5, 4.5]
        ),
        hovermode="x unified",
        template="plotly_white",
        height=550
    )

    fig.show()


In [12]:
transition_rows = []

for city, (model, forecast) in forecast_results.items():

    df_f = forecast.copy()
    df_f["Year"] = df_f["ds"].dt.year
    df_f["HSI_Level"] = df_f["yhat"].round().clip(0, 4)

    # Current HSI (latest historical)
    hist = model.history.copy()
    hist = hist.sort_values("ds")
    current_hsi = int(round(hist.iloc[-1]["y"]))

    # First sustainable year
    sustainable_years = df_f[df_f["HSI_Level"] >= 3]["Year"]
    sustainable_year = sustainable_years.min() if not sustainable_years.empty else 2030

    transition_rows.append({
        "City": city,
        "Current_HSI": current_hsi,
        "Sustainable_Year": sustainable_year
    })

transition_df = pd.DataFrame(transition_rows)


In [14]:
hsi_labels = {
    0: "Critical",
    1: "Low Sustainable",
    2: "Moderately Sustainable",
    3: "Sustainable",
    4: "Highly Sustainable"
}

hsi_colors = {
    0: "#d73027",   # Red
    1: "#fc8d59",   # Orange
    2: "#fee08b",   # Yellow
    3: "#1a9850",   # Green
    4: "#4575b4"    # Blue
}

transition_df["Current_Status"] = transition_df["Current_HSI"].map(hsi_labels)
transition_df["Sustainable_Label"] = transition_df["Sustainable_Year"].replace(
    {2030: "Beyond 2029"}
)


In [32]:
import plotly.graph_objects as go

fig = go.Figure()
maintenance_year = 2030  # Where the maintenance text appears

for _, row in transition_df.iterrows():
    city = row["City"]
    curr_hsi = row["Current_HSI"]
    target_year = row["Sustainable_Year"]
    is_sustainable = curr_hsi >= 3

    # 1. ADDING THE LINE: Only for cities NOT yet sustainable
    if not is_sustainable:
        fig.add_shape(
            type="line",
            x0=2024, y0=city, x1=target_year, y1=city,
            line=dict(color="gray", dash="dot", width=1.5)
        )

    # 2. CURRENT STATUS: Square Marker
    fig.add_trace(go.Scatter(
        x=[2024], y=[city],
        mode="markers+text",
        marker=dict(size=18, color=hsi_colors[curr_hsi], symbol="square"),
        text=f"<b>{row['Current_Status']}</b>",
        textposition="middle right",
        showlegend=False
    ))

    # 3. INDIVIDUAL LOGIC: If Sustainable, show the Maintenance Text
    if is_sustainable:
        fig.add_trace(go.Scatter(
            x=[maintenance_year], y=[city],
            mode="markers+text",
            marker=dict(size=16, symbol="diamond", color="#1a9850"),
            text="<b>Projected to Remain Sustainable</b>",
            textposition="middle right",
            name="Forecast: Year When City Reaches Sustainable Level",
            showlegend=True # Show legend once for the sustainable city
        ))
    else:
        # For non-sustainable cities, show the date label
        fig.add_trace(go.Scatter(
            x=[target_year], y=[city],
            mode="markers+text",
            marker=dict(size=16, symbol="diamond", color="#1a9850"),
            text=f" {row['Sustainable_Label']}",
            textposition="middle right",
            name="Forecast: Year When City Reaches Sustainable Level",
            showlegend=False # Hide to avoid duplicate legend entries
        ))

# ---- Layout ----
fig.update_layout(
    title="<b>City-wise Human Sustainability Index: Current Status & It's Forecasting</b>",
    xaxis=dict(
        tickvals=[2024, 2025, 2026, 2027, 2028, 2029, 2030],
        ticktext=["2024", "2025", "2026", "2027", "2028", "2029", "2030+"],
        range=[2022, 2036] # Extra space for the long "Maintain" text
    ),
    template="plotly_white",
    height=465,
    width=980,
    margin=dict(r=180), # Increased margin so text isn't cut off
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()